In [2]:
# ============================================================
# Toronto Restaurant Risk & Compliance Intelligence System
# Notebook 01: Data Validation & QA
# Author: Sreekar Koduru
# Date: June 2026
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded ✅")
print(f"Pandas version: {pd.__version__}")
print(f"Analysis date: {datetime.today().strftime('%Y-%m-%d')}")

Libraries loaded ✅
Pandas version: 2.2.2
Analysis date: 2026-06-22


In [6]:
# ============================================================
# STEP 1: Load Dataset
# ============================================================

import os

# Use absolute path — works regardless of where Jupyter is launched from
RAW_DATA_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/data/raw/Dinesafe.csv'
)

# Verify file exists before loading
print(f"File exists: {os.path.exists(RAW_DATA_PATH)}")
print(f"Full path: {RAW_DATA_PATH}")

# Load dataset
df = pd.read_csv(RAW_DATA_PATH)

print("\n=== DATASET LOADED ✅ ===")
print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

File exists: True
Full path: /Users/sreekarkoduru/Projects/project3-toronto-restaurant-compliance/data/raw/Dinesafe.csv

=== DATASET LOADED ✅ ===
Rows:    102,766
Columns: 18

Column names:
  - _id
  - unique_id
  - estId
  - oldEstId
  - estName
  - address
  - inspectionStatus
  - phone
  - inspectionDate
  - observation
  - typeDesc
  - deficiencyDesc
  - severity
  - OutcomeDate
  - OutcomeDesc
  - amountFined
  - latitude
  - longitude


In [8]:
# ============================================================
# STEP 2: Data Validation & QA Report
# ============================================================

print("=" * 60)
print("TORONTO DINESAFE — DATA VALIDATION & QA REPORT")
print(f"Analysis Date: {datetime.today().strftime('%Y-%m-%d')}")
print("=" * 60)

# ----------------------------------------------------------
# CHECK 1: Row and Column Count
# ----------------------------------------------------------
print("\n📋 CHECK 1: Dataset Dimensions")
print(f"  Total rows:    {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Result: ✅ PASS")

# ----------------------------------------------------------
# CHECK 2: Duplicate Records
# ----------------------------------------------------------
print("\n📋 CHECK 2: Duplicate Records")
total_dupes = df.duplicated().sum()
unique_id_dupes = df['unique_id'].duplicated().sum()
print(f"  Fully duplicate rows:      {total_dupes:,}")
print(f"  Duplicate unique_id values: {unique_id_dupes:,}")
if total_dupes == 0 and unique_id_dupes == 0:
    print(f"  Result: ✅ PASS — No duplicates found")
else:
    print(f"  Result: ⚠️ ACTION REQUIRED — Duplicates found")

# ----------------------------------------------------------
# CHECK 3: Null Values
# ----------------------------------------------------------
print("\n📋 CHECK 3: Null Values per Column")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct,
    'Status': null_pct.apply(
        lambda x: '✅ OK' if x == 0 
        else '⚠️ HIGH' if x > 50 
        else '📋 NOTE'
    )
})
print(null_df.to_string())

# ----------------------------------------------------------
# CHECK 4: Date Range Validation
# ----------------------------------------------------------
print("\n📋 CHECK 4: Date Range Validation")
df['inspectionDate'] = pd.to_datetime(df['inspectionDate'], errors='coerce')
invalid_dates = df['inspectionDate'].isnull().sum()
future_dates = (df['inspectionDate'] > datetime.today()).sum()
print(f"  Earliest inspection: {df['inspectionDate'].min().date()}")
print(f"  Latest inspection:   {df['inspectionDate'].max().date()}")
print(f"  Invalid dates:       {invalid_dates:,}")
print(f"  Future dates:        {future_dates:,}")
if invalid_dates == 0:
    print(f"  Result: ✅ PASS — All dates valid")
else:
    print(f"  Result: ⚠️ ACTION REQUIRED — {invalid_dates} invalid dates")

# ----------------------------------------------------------
# CHECK 5: Categorical Field Validation
# ----------------------------------------------------------
print("\n📋 CHECK 5: Categorical Field Validation")

print("\n  inspectionStatus values:")
for val, count in df['inspectionStatus'].value_counts().items():
    pct = count/len(df)*100
    print(f"    {val:<20} {count:>7,} ({pct:.1f}%)")

print("\n  severity values (excluding nulls):")
for val, count in df['severity'].value_counts().items():
    pct = count/len(df)*100
    print(f"    {val:<20} {count:>7,} ({pct:.1f}%)")

# Check for unexpected values
valid_status = {'Pass', 'Conditional Pass', 'Closed'}
valid_severity = {'M - Minor', 'S - Significant', 'C - Crucial'}
unexpected_status = set(df['inspectionStatus'].unique()) - valid_status
unexpected_severity = set(df['severity'].dropna().unique()) - valid_severity
print(f"\n  Unexpected inspectionStatus values: {unexpected_status if unexpected_status else 'None ✅'}")
print(f"  Unexpected severity values:         {unexpected_severity if unexpected_severity else 'None ✅'}")

# ----------------------------------------------------------
# CHECK 6: Coordinate Validation
# ----------------------------------------------------------
print("\n📋 CHECK 6: Coordinate Validation")
lat_nulls = df['latitude'].isnull().sum()
lon_nulls = df['longitude'].isnull().sum()
invalid_lat = ((df['latitude'] < 43.58) | (df['latitude'] > 43.85)).sum()
invalid_lon = ((df['longitude'] < -79.64) | (df['longitude'] > -79.12)).sum()
print(f"  Null latitudes:      {lat_nulls:,}")
print(f"  Null longitudes:     {lon_nulls:,}")
print(f"  Invalid latitudes:   {invalid_lat:,}")
print(f"  Invalid longitudes:  {invalid_lon:,}")
print(f"  Lat range: {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
print(f"  Lon range: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")
if invalid_lat == 0 and invalid_lon == 0:
    print(f"  Result: ✅ PASS — All coordinates within Toronto bounds")
else:
    print(f"  Result: ⚠️ ACTION REQUIRED — Invalid coordinates found")

# ----------------------------------------------------------
# CHECK 7: Fine Amount Validation
# ----------------------------------------------------------
print("\n📋 CHECK 7: Fine Amount Validation")
fine_records = df['amountFined'].notna().sum()
fine_stats = df['amountFined'].describe()
negative_fines = (df['amountFined'] < 0).sum()
print(f"  Records with fines:  {fine_records:,}")
print(f"  Records without:     {df['amountFined'].isna().sum():,}")
print(f"  Mean fine:           ${fine_stats['mean']:,.2f}")
print(f"  Max fine:            ${fine_stats['max']:,.2f}")
print(f"  Negative fines:      {negative_fines:,}")
if negative_fines == 0:
    print(f"  Result: ✅ PASS — No negative fine amounts")

# ----------------------------------------------------------
# CHECK 8: Establishment ID Consistency
# ----------------------------------------------------------
print("\n📋 CHECK 8: Establishment ID Consistency")
unique_est_ids = df['estId'].nunique()
unique_est_names = df['estName'].nunique()
print(f"  Unique estId values:   {unique_est_ids:,}")
print(f"  Unique estName values: {unique_est_names:,}")
print(f"  ID vs Name difference: {unique_est_ids - unique_est_names:,}")
print(f"  Note: Difference indicates name changes or inconsistencies")
print(f"  Action: Use estId for grouping, not estName ✅")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

TORONTO DINESAFE — DATA VALIDATION & QA REPORT
Analysis Date: 2026-06-22

📋 CHECK 1: Dataset Dimensions
  Total rows:    102,766
  Total columns: 18
  Result: ✅ PASS

📋 CHECK 2: Duplicate Records
  Fully duplicate rows:      0
  Duplicate unique_id values: 0
  Result: ✅ PASS — No duplicates found

📋 CHECK 3: Null Values per Column
                  Null Count  Null %   Status
_id                        0    0.00     ✅ OK
unique_id                  0    0.00     ✅ OK
estId                      0    0.00     ✅ OK
oldEstId                3059    3.00   📋 NOTE
estName                    0    0.00     ✅ OK
address                    0    0.00     ✅ OK
inspectionStatus           0    0.00     ✅ OK
phone                  13036   12.70   📋 NOTE
inspectionDate             0    0.00     ✅ OK
observation                0    0.00     ✅ OK
typeDesc               37559   36.50   📋 NOTE
deficiencyDesc         37559   36.50   📋 NOTE
severity               42076   40.90   📋 NOTE
OutcomeDate           1

In [10]:
# ============================================================
# STEP 3: Validation Summary Table
# ============================================================

print("=" * 60)
print("VALIDATION SUMMARY — ACTION TABLE")
print("=" * 60)

validation_summary = pd.DataFrame([
    {
        'Check': 'Duplicate Records',
        'Result': '0 duplicates',
        'Action Taken': 'No action needed',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Null Values — phone',
        'Result': '13,036 (12.7%)',
        'Action Taken': 'Excluded from analysis — not used',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Null Values — typeDesc / deficiencyDesc',
        'Result': '37,559 (36.5%)',
        'Action Taken': 'Expected — Pass inspections have no violations',
        'Status': '📋 NOTE'
    },
    {
        'Check': 'Null Values — severity',
        'Result': '42,076 (40.9%)',
        'Action Taken': 'Expected — nulls = clean inspections. Risk score = 0 for these records',
        'Status': '📋 NOTE'
    },
    {
        'Check': 'Null Values — OutcomeDate / OutcomeDesc / amountFined',
        'Result': '~99.8% null',
        'Action Taken': 'Expected — enforcement actions are rare. Fine analysis uses 182 non-null records only',
        'Status': '📋 NOTE'
    },
    {
        'Check': 'Invalid Dates',
        'Result': '0 invalid dates',
        'Action Taken': 'No action needed',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Future Dates',
        'Result': '0 future dates',
        'Action Taken': 'No action needed',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Unexpected Category Values',
        'Result': 'None found',
        'Action Taken': 'No action needed',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Coordinate Validation',
        'Result': '0 null, 0 invalid',
        'Action Taken': 'No action needed — all within Toronto bounds',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Negative Fine Amounts',
        'Result': '0 negative values',
        'Action Taken': 'No action needed',
        'Status': '✅ PASS'
    },
    {
        'Check': 'Establishment ID vs Name Mismatch',
        'Result': '3,698 difference',
        'Action Taken': 'Use estId for all grouping — not estName',
        'Status': '📋 NOTE'
    },
])

# Display
for _, row in validation_summary.iterrows():
    print(f"\n{row['Status']} {row['Check']}")
    print(f"   Result: {row['Result']}")
    print(f"   Action: {row['Action Taken']}")

print("\n" + "=" * 60)
print("DATA QUALITY SCORE")
print("=" * 60)

# Calculate quality score
total_checks = len(validation_summary)
passed = len(validation_summary[validation_summary['Status'] == '✅ PASS'])
noted = len(validation_summary[validation_summary['Status'] == '📋 NOTE'])
failed = len(validation_summary[validation_summary['Status'] == '❌ FAIL'])

print(f"  Total checks run:  {total_checks}")
print(f"  Passed:            {passed}")
print(f"  Notes (expected):  {noted}")
print(f"  Failed:            {failed}")
print(f"\n  Data Quality Score: {passed/total_checks*100:.0f}%")
print(f"  Assessment: Data is fit for analysis with documented assumptions")

# Save validation summary to CSV
VALIDATION_OUTPUT = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/documents/validation_summary.csv'
)
validation_summary.to_csv(VALIDATION_OUTPUT, index=False)
print(f"\n  Validation summary saved to: documents/validation_summary.csv ✅")

VALIDATION SUMMARY — ACTION TABLE

✅ PASS Duplicate Records
   Result: 0 duplicates
   Action: No action needed

✅ PASS Null Values — phone
   Result: 13,036 (12.7%)
   Action: Excluded from analysis — not used

📋 NOTE Null Values — typeDesc / deficiencyDesc
   Result: 37,559 (36.5%)
   Action: Expected — Pass inspections have no violations

📋 NOTE Null Values — severity
   Result: 42,076 (40.9%)
   Action: Expected — nulls = clean inspections. Risk score = 0 for these records

📋 NOTE Null Values — OutcomeDate / OutcomeDesc / amountFined
   Result: ~99.8% null
   Action: Expected — enforcement actions are rare. Fine analysis uses 182 non-null records only

✅ PASS Invalid Dates
   Result: 0 invalid dates
   Action: No action needed

✅ PASS Future Dates
   Result: 0 future dates
   Action: No action needed

✅ PASS Unexpected Category Values
   Result: None found
   Action: No action needed

✅ PASS Coordinate Validation
   Result: 0 null, 0 invalid
   Action: No action needed — all within

In [12]:
# Fixed quality score — Notes are not failures
print("=" * 60)
print("DATA QUALITY SCORE (CORRECTED)")
print("=" * 60)

total_checks = 11
passed = 7
notes = 4  # Expected patterns — not errors
failed = 0

# Score = (Passed + Notes) / Total — Notes are acceptable
adjusted_score = (passed + notes) / total_checks * 100

print(f"  Passed (no action needed): {passed}")
print(f"  Notes (expected patterns): {notes}")
print(f"  Failed (errors):           {failed}")
print(f"\n  Adjusted Data Quality Score: {adjusted_score:.0f}%")
print(f"  Assessment: Data is fit for analysis.")
print(f"  All issues identified are expected data characteristics,")
print(f"  not errors. No records need to be excluded at this stage.")

DATA QUALITY SCORE (CORRECTED)
  Passed (no action needed): 7
  Notes (expected patterns): 4
  Failed (errors):           0

  Adjusted Data Quality Score: 100%
  Assessment: Data is fit for analysis.
  All issues identified are expected data characteristics,
  not errors. No records need to be excluded at this stage.
